# 01 - Data and the training graph

A short tour of the released DisAtlas inputs: the disease-gene table (public sources plus the 8,343 released Habcd knowledge-graph edges), the **broad** channel built from it (shared genes and phenotypes), the released **specific** channel edges (shared mechanism modules), the P_full training graph rebuilt from them, and pair scores from the five-seed ensemble.

Run from the repository root after fetching the Zenodo bundle into `data/zenodo` (see `data/README.md`). Only repository paths are read.

In [ ]:
import os
if not os.path.isdir('data') and os.path.isdir('../data'):
    os.chdir('..')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from disatlas.graph import raw_build, pfull_build

## Disease-gene table

The broad channel's disease-gene table is rebuilt from the public sources in `data/raw` (Open Targets 26.09 direct, GenCC, Orphanet, HPO-derived annotations) and the released Habcd edges (`data/raw/bikg_habcd`).

In [ ]:
dg = raw_build.gene_edges()
rel = pd.read_parquet('data/raw/bikg_habcd/habcd_bikg_gene_edges.parquet')
key = lambda d: set(zip(d.disease_id, d.gene_symbol))
t1 = dg[dg.evidence_tier == 1]
pd.Series({'disease-gene rows': len(dg), 'diseases': dg.disease_id.nunique(), 'genes': dg.gene_symbol.nunique(),
           'tier-1 rows': len(t1), 'tier-1 rows also in the Habcd release': len(key(t1) & key(rel)),
           'released Habcd edges': len(rel)}).to_frame('n')

In [ ]:
dg.evidence_tier.value_counts().sort_index().rename('rows').to_frame()

## Channels and training graph

The broad graph ships in `data/zenodo/graph/graph_edges.parquet` (rebuilt from `data/raw` by `python -m disatlas.graph.raw_build`); the specific channel ships as edges only (`data/raw/channels/specific/specific_edges.parquet`). `pfull_build.build` combines them into the P_full training graph and checks its signature.

In [ ]:
broad = pd.read_parquet('data/zenodo/graph/graph_edges.parquet')
spec = pd.read_parquet('data/raw/channels/specific/specific_edges.parquet')
graph, stats = pfull_build.build()
def n_nodes(e): return pd.concat([e.u, e.v]).nunique()
pd.DataFrame({'edges': [len(broad), len(spec), len(graph)], 'diseases': [n_nodes(broad), n_nodes(spec), n_nodes(graph)]},
             index=['broad channel', 'specific channel', 'P_full training graph'])

In [ ]:
ov = graph.assign(channel=np.select([graph.in_broad & graph.in_spec, graph.in_broad], ['both', 'broad only'], 'specific only'))
ov.channel.value_counts().rename('edges').to_frame()

## Degree distributions

In [ ]:
def degree(e): return pd.concat([e.u, e.v]).value_counts()
fig, ax = plt.subplots(figsize=(6, 3.5))
for name, e in [('broad', broad), ('specific', spec), ('training graph', graph)]:
    d = degree(e)
    ax.hist(d, bins=np.logspace(0, np.log10(d.max()), 40), histtype='step', label=f'{name} (median {int(d.median())})')
ax.set(xscale='log', yscale='log', xlabel='Disease degree', ylabel='Diseases')
ax.legend(frameon=False); fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(graph.w_arm, bins=60)
ax.set(yscale='log', xlabel='Walk weight (w_arm)', ylabel='Edges'); fig.tight_layout()

## Ensemble pair scores

DisAtlas scores a disease pair by the cosine of the two 32-dimensional disease embeddings of the final (gene-anchored) model, averaged over the five seeds (4707-4711); the clustering distance is one minus this cosine. Below: the highest-cosine partners of a few example diseases, and the cosine of some example pairs.

In [ ]:
from disatlas.model import load_ensemble
m = load_ensemble()
examples = ['MONDO:0005148', 'MONDO:0004975', 'MONDO:0007915']  # type 2 diabetes, Alzheimer disease, SLE
examples = [x for x in examples if x in set(m.ids)]
pd.concat({d: m.top_partners(d, k=5) for d in examples})

In [ ]:
pairs = pd.DataFrame({'disease_a': ['MONDO:0005148', 'MONDO:0007915', 'MONDO:0005148'],
                      'disease_b': ['MONDO:0005015', 'MONDO:0008383', 'MONDO:0004975']})
pairs['cosine'] = m.score_pairs(pairs.disease_a, pairs.disease_b)
pairs